# Tiny LLM Decoder 2 (Pytorch Implementation Line by Line)

In the second version we are converting everything into Pytorch so that we can perform back propagation using Pytorch.

In [1]:
import numpy as np
import scipy.stats as stats
import torch
import torch.nn as nn
from torch.nn import functional as F

## LLM Architecture

At high level, the training process is as follows:

```text

Text_processing -> Transformer Network -> Compare Loss -> Back Propagation
```

The inference process is as follows:

```text
Text_processing -> Transformer Network -> Predict Next Token -> Repeat until token size reached
```

### Corpus

In [2]:
# Text corpus
corpus = """
mary had a little lamb
its fleece was white as snow
and everywhere that mary went
the lamb was sure to go
mary had a little lamb
the lamb was sure to go
"""


### Parameters

In [3]:
# Set Parameters

# d_model is embedding dimension
d_model = 4

# block_size is the same as context window or maximum sequence length
block_size = 5


## Text Processing

### Setup

In [4]:
# Tokenization

# Split corpus into words
word_tokens = corpus.split()

# Use set to isolate unique words
vocab = sorted(list(set(word_tokens)))
vocab_size = len(vocab)

# Create token id and create lookup table from word to token id
# i is the index or tken id in this case, so we assign token id using sequential number
words_to_id_ref = {word: i for i, word in enumerate(vocab)}

# Create lookup table from id to words
id_to_words_ref = {i: word for i, word in enumerate(vocab)}

In [5]:
vocab_size

19

In [6]:
# convert corpus to id
corpus_in_id = [words_to_id_ref[word] for word in word_tokens]


In [7]:
# Setup embeddings initialized randomly
random_seed = 42
embeddings = torch.nn.Embedding(vocab_size, d_model)
embeddings


Embedding(19, 4)

In [8]:
embeddings.weight

Parameter containing:
tensor([[-1.5710,  1.0093, -0.5375, -0.2165],
        [ 0.2880,  0.6352,  2.4762,  1.1328],
        [-1.5404, -0.5349,  0.3822,  0.7416],
        [-0.9434, -0.5756,  1.0042,  2.4998],
        [ 1.0206, -0.9649, -0.4577, -0.5354],
        [ 0.8864, -1.8737,  0.4160, -0.8867],
        [-1.2436,  0.0182,  0.5067, -0.7496],
        [-0.9711,  0.6627,  1.5568, -1.2816],
        [-0.6553,  1.4045, -0.4679, -0.8654],
        [ 1.3580, -1.5861,  0.9854, -0.3863],
        [ 1.0636, -0.1321,  0.0382,  1.0643],
        [-1.3313, -1.6388, -0.6750, -0.4143],
        [-1.4442,  0.1390, -0.8999, -0.1706],
        [-0.6106, -0.6815,  0.8813,  0.2069],
        [-0.8828, -0.6950, -0.2354,  1.5999],
        [-0.3031, -1.8076,  0.3745, -1.7156],
        [-1.4668, -0.2768, -0.0766,  0.3389],
        [-0.2133,  1.9912, -0.4465, -0.9022],
        [ 1.0152, -0.4277, -1.5122, -2.3713]], requires_grad=True)

### Text Processing for Training

In [9]:
def get_sequence_block(block_size, start_position, corpus_in_id):
  """
  This function extract the block in id from the corpus in id
  User need to supply the block size and the start position
  Return: X, Y, next_start_position
  """
  X = corpus_in_id[start_position: start_position+block_size]
  Y = corpus_in_id[start_position+1: start_position+block_size+1]
  next_start_position = start_position + block_size
  return X, Y, next_start_position

In [10]:
def get_N(corpus_in_id, block_size):
  """
  This function calculate the number of block we need to run for the entire corpus
  """
  N = (len(corpus_in_id)-1) // block_size
  return N

In [11]:
N = get_N(corpus_in_id, block_size=block_size)
N

6

In [12]:
X, Y, next_start = get_sequence_block(block_size=block_size, start_position=0, corpus_in_id=corpus_in_id)
X, Y, next_start

([10, 6, 0, 9, 8], [6, 0, 9, 8, 7], 5)

In [13]:
# Get embedding based on X token ID
X_input = embeddings(torch.tensor(X, dtype=torch.long))
X_input

tensor([[ 1.0636, -0.1321,  0.0382,  1.0643],
        [-1.2436,  0.0182,  0.5067, -0.7496],
        [-1.5710,  1.0093, -0.5375, -0.2165],
        [ 1.3580, -1.5861,  0.9854, -0.3863],
        [-0.6553,  1.4045, -0.4679, -0.8654]], grad_fn=<EmbeddingBackward0>)

In [14]:
# Get positional Endccoding
position = np.arange(0,block_size)
position = position.reshape(-1,1)
position = torch.Tensor(position)
position

tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.]])

In [15]:
X_input_pos = X_input + position
X_input_pos

tensor([[ 1.0636, -0.1321,  0.0382,  1.0643],
        [-0.2436,  1.0182,  1.5067,  0.2504],
        [ 0.4290,  3.0093,  1.4625,  1.7835],
        [ 4.3580,  1.4139,  3.9854,  2.6137],
        [ 3.3447,  5.4045,  3.5321,  3.1346]], grad_fn=<AddBackward0>)

In [16]:
X_input_pos.shape

torch.Size([5, 4])

In [17]:
print(X_input_pos.requires_grad)
print(X_input_pos.shape)

True
torch.Size([5, 4])


The X_input is ready to feed into the transformer network. Lets assign it to X for simplicity.

In [18]:
X = X_input_pos

## Transformer

### Attention Formula

We will be implementing the attention formula so we need to understand it

$$\text{Attention (Q,K,V)} = \text{Softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right) V$$


- $Q$ refers to the token we are asking or checking, in our first block `mary` will be the Q.

- $K$ refers to all other words in the block. In our context it will be `mary had a little lamb`. We will need to check `mary` against `mary`.

- $V$ refers to the value.

- $QK$ we multiple $Q$ by $K$ to get the relative position between $Q$ and each $K$. The $T$ refers to transpose, it is a necessary step in doing matrix multiplication.

- $d_k$ is a normalization factor so that the numbers $QK$ are stable. In our case $d_k$ is the model dimension.




### Simple Architecture

In this architecture, we use the simplest architecture

```text

input text pre-processing (X)
      |
      V
Self attention
      |
      V
Residual layer
      |
      V
Normalized Layer
      |
      V
Feed Forward Network
      |
      V
Residual Layer
      |
      V
Normalized Layer

```

In Feed Forward Network, we have 3 layers

```text
Linear Layer
      |
      V
ReLU Layer
      |
      V
Linear Layer

```

## Self Attention Layer

In [19]:
# Create a random generator with a random seed
rnd = np.random.default_rng(random_seed)


In [20]:
# Initializing weights (4,4) for WQ, WK and WV
WQ = nn.Parameter(torch.Tensor(rnd.standard_normal((d_model, d_model))))
WK = nn.Parameter(torch.Tensor(rnd.standard_normal((d_model, d_model))))
WV = nn.Parameter(torch.Tensor(rnd.standard_normal((d_model, d_model))))

In [21]:
print(WQ.requires_grad)
print(WK.requires_grad)
print(WV.requires_grad)

True
True
True


In [22]:
WQ

Parameter containing:
tensor([[ 0.3047, -1.0400,  0.7505,  0.9406],
        [-1.9510, -1.3022,  0.1278, -0.3162],
        [-0.0168, -0.8530,  0.8794,  0.7778],
        [ 0.0660,  1.1272,  0.4675, -0.8593]], requires_grad=True)

In [23]:
WK

Parameter containing:
tensor([[ 0.3688, -0.9589,  0.8785, -0.0499],
        [-0.1849, -0.6809,  1.2225, -0.1545],
        [-0.4283, -0.3521,  0.5323,  0.3654],
        [ 0.4127,  0.4308,  2.1416, -0.4064]], requires_grad=True)

In [24]:
# Calculate QKV
# Q = X @ WQ
Q = X @ WQ
K = X @ WK
V = X @ WV

In [25]:
Q.shape

torch.Size([5, 4])

In [26]:
# Compute attention score
attention_score = Q @ K.T
attention_score.shape

torch.Size([5, 5])

In [27]:
# Divide by sqrt(d_k)
stable_factor = np.sqrt(d_model)
attention_stable = attention_score / stable_factor
attention_stable.shape

torch.Size([5, 5])

## Causal Masking

In [28]:
causal_mask = np.triu(np.ones(attention_stable.shape), k=1)
causal_mask

array([[0., 1., 1., 1., 1.],
       [0., 0., 1., 1., 1.],
       [0., 0., 0., 1., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0.]])

In [29]:
# actual causal masking
causal_mask = torch.triu(
    torch.full_like(attention_stable, float("-inf")),
    diagonal=1
)
causal_mask.shape

torch.Size([5, 5])

In [30]:
attention_stable

tensor([[  2.1978,   1.2086,   5.2745,   8.3108,  11.0402],
        [  1.6774,   3.4720,   8.4638,  14.0963,  19.4232],
        [  3.0576,   7.1099,  17.3384,  26.5154,  38.2679],
        [ 12.6452,  13.9273,  41.5852,  72.2119,  95.0836],
        [ 10.0906,  17.6893,  45.5096,  75.2818, 103.5750]],
       grad_fn=<DivBackward0>)

In [31]:
attention_stable_mask = attention_stable + causal_mask
attention_stable_mask

tensor([[  2.1978,     -inf,     -inf,     -inf,     -inf],
        [  1.6774,   3.4720,     -inf,     -inf,     -inf],
        [  3.0576,   7.1099,  17.3384,     -inf,     -inf],
        [ 12.6452,  13.9273,  41.5852,  72.2119,     -inf],
        [ 10.0906,  17.6893,  45.5096,  75.2818, 103.5750]],
       grad_fn=<AddBackward0>)

In [32]:
attention_stable_mask.shape

torch.Size([5, 5])

In this 5 by 5 matrix we have

| col | mary | had | a | little | lamb |
| --- | --- | --- | --- | --- | --- |
| mary | 0.17 | -inf | -inf | -inf | -inf |
| had | 1.9 | 6.8 | -inf | -inf | -inf |
| a | 4.5 | 15.3 | 33.67 | -inf | -inf |
| little | 4.75 | 16.8 | 36.53 | 46.7 | -inf |
| lamb | 7.45 | 26.5 | 57.5 | 73.38 | 108.5 |


So `mary` can only see `mary` and it need to predict the next token. The second last row `mary had a little` is given but `lamb` is masked. the model suppose to predict mask in the 4th row.

In [33]:
attention_softmax = torch.softmax(attention_stable_mask, dim=-1)
attention_softmax.shape

torch.Size([5, 5])

In [34]:
attention_softmax

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.4251e-01, 8.5749e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [6.2789e-07, 3.6124e-05, 9.9996e-01, 0.0000e+00, 0.0000e+00],
        [1.3506e-26, 4.8679e-26, 5.0002e-14, 1.0000e+00, 0.0000e+00],
        [2.5135e-41, 5.0159e-38, 6.0609e-26, 5.1574e-13, 1.0000e+00]],
       grad_fn=<SoftmaxBackward0>)

In [35]:
attention_output = attention_softmax @ V
attention_output.shape

torch.Size([5, 4])

In [36]:
attention_output

tensor([[-0.3772, -0.5010,  1.6662,  1.3617],
        [ 0.9391,  0.1137, -1.2838,  0.8743],
        [ 0.7325, -1.6929, -1.6359,  3.1803],
        [ 0.8736, -1.9981,  1.1440,  7.3496],
        [ 0.6619, -4.6585, -2.0148,  8.8131]], grad_fn=<MmBackward0>)

### Residual Layer

In [37]:
attention_output_residual = attention_output + X
attention_output_residual.shape

torch.Size([5, 4])

### Normalized Layer

In [38]:
norm1 = nn.LayerNorm(d_model)
attention_output_residual_scaled = norm1(attention_output_residual)
attention_output_residual_scaled.shape

torch.Size([5, 4])

### Feed Forward Layer (FFN)

In [39]:
ffn = nn.Sequential(
    nn.Linear(d_model, d_model, bias=False),
    nn.ReLU(),
    nn.Linear(d_model, d_model, bias=False)
)

In [40]:
# FFN 1 - Linear Layer
ffn_1 = ffn(attention_output_residual_scaled)
ffn_1.shape

torch.Size([5, 4])

### Second Residual Layer

In [41]:
seond_residual = ffn_1 + attention_output_residual_scaled
seond_residual.shape

torch.Size([5, 4])

### Second Normalized Layer

In [42]:
# 2nd layer
norm2 = nn.LayerNorm(d_model)
second_norm = norm2(seond_residual)
second_norm.shape

torch.Size([5, 4])

In [43]:
transformer_output = second_norm
transformer_output.shape

torch.Size([5, 4])

### Convert to Logits

In [44]:
embeddings.weight.shape

torch.Size([19, 4])

In [45]:
logits = transformer_output @ embeddings.weight.T
logits.shape

torch.Size([5, 19])

In [46]:
logits

tensor([[-1.1808,  1.8164,  2.6233,  4.9867, -0.1251,  1.1967, -0.1092, -1.2154,
         -2.9483,  1.5358,  1.0524,  1.9513, -0.2067,  1.9635,  3.2841,  0.5940,
          1.4711, -3.9885, -3.7807],
        [ 1.9928, -1.9845,  0.4432,  1.1310, -1.0927, -3.3907, -0.9988, -2.6768,
          1.1423, -3.6473,  0.5301, -0.2191,  1.8208, -1.3414,  1.8453, -3.6674,
          0.8527,  1.3564, -1.0882],
        [ 0.4152, -0.2664,  2.1005,  4.3777, -0.7170, -1.4573, -0.9775, -3.0914,
         -1.4400, -1.3933,  1.2822,  1.2532,  1.0821,  0.3759,  3.7330, -2.2313,
          1.5751, -2.0314, -3.3727],
        [-1.5118,  1.0370,  2.1237,  4.6624,  0.3342,  1.2499, -0.7963, -2.3419,
         -3.1338,  1.5607,  1.5129,  1.8253, -0.3125,  1.4776,  3.3585,  0.1848,
          1.1131, -4.0761, -3.1725],
        [-0.5116,  0.2348,  2.1582,  4.6994, -0.2020, -0.2527, -1.0299, -3.0421,
         -2.3356, -0.0728,  1.5157,  1.5720,  0.4690,  0.8527,  3.7611, -1.2625,
          1.3949, -3.1107, -3.3780]], grad

### Calculate Loss

In [47]:
criterion = nn.CrossEntropyLoss()

## Linking Everything - Forward Pass

In [48]:
def forward(X):
    # Word embeddings + positional encoding
    X_input = embeddings(X)
    position = torch.arange(len(X), dtype=X_input.dtype).reshape(-1, 1)
    X_input_pos = X_input + position

    # Q, K, V projections
    Q = X_input @ WQ
    K = X_input @ WK
    V = X_input @ WV

    # Scaled dot-product attention
    attention_score = Q @ K.T
    stable_factor = np.sqrt(d_model)
    attention_stable = attention_score / stable_factor

    # Causal masking + softmax
    causal_mask = torch.triu(
        torch.full_like(attention_stable, float("-inf")),
        diagonal=1
    )
    attention_stable_mask = attention_stable + causal_mask
    attention_softmax = torch.softmax(attention_stable_mask, dim=-1)

    # Attention output + residual + norm1
    attention_output = attention_softmax @ V
    attention_output_residual = attention_output + X_input
    norm1 = nn.LayerNorm(d_model)
    attention_output_residual_scaled = norm1(attention_output_residual)

    # FFN + residual + norm2
    ffn = nn.Sequential(
        nn.Linear(d_model, d_model, bias=False),
        nn.ReLU(),
        nn.Linear(d_model, d_model, bias=False)
    )
    ffn_1 = ffn(attention_output_residual_scaled)
    seond_residual = ffn_1 + attention_output_residual_scaled
    norm2 = nn.LayerNorm(d_model)
    transformer_output = norm2(seond_residual)

    # Vocabulary projection
    logits = transformer_output @ embeddings.weight.T

    return logits

We include embeddings lookup and positional encoding in the forward loop, thus we need to extract the first block and feed into the forward function.

> During implementation `X_input = embeddings(X)` was used because we already convert to tensor before feeding in to teh function.

> We also change the code `position = torch.arange(len(X), dtype=X_input.dtype).reshape(-1, 1)` to be pytorch friendly.

In [49]:
X, Y, next_start = get_sequence_block(block_size=block_size, start_position=0, corpus_in_id=corpus_in_id)
X = torch.tensor(X, dtype=torch.long)
logits = forward(X)
logits.shape

torch.Size([5, 19])

The code above is in preparation for a training loop with back propagation.

## END